In [43]:
import os
import certifi
# import requests
from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub
from langchain.tools import tool

In [14]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [16]:
%pip install -U langchain-ollama


   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

response = llm.invoke("What is an AI agent?")
print(response.content)

An AI agent is a software program or a machine that is designed to perceive its environment, understand the context, and take actions to achieve its goals. These agents are often used in various applications such as robotics, gaming, and automation. They can be programmed to learn from experience, adapt to new situations, and interact with humans and other agents in a way that mimics human behavior.

AI agents can be categorized into different types based on their capabilities and the environment they operate in:

1. **Mobile Agents**: These are software agents that can move from one computer to another, often used in distributed computing environments.

2. **Service Agents**: These are designed to provide services to other agents or users, often found in service-oriented architectures.

3. **Social Agents**: These are designed to interact with humans in a way that is similar to human-human interactions, often used in virtual assistants, chatbots, and recommendation systems.

4. **Auto

In [18]:
os.environ["SSL_CERY_FILE"] = certifi.where()
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [19]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\sajal\AppData\Local\Temp\ipykernel_14896\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [20]:
result = search_tool.invoke("what is the lastest news on AI")
result

[{'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com',
  'url': 'https://www.wsj.com/tech/ai',
  'content': '21\n\nTemasek-backed Vertex Ventures Japan has invested in Heartbeats, a Tokyo-based digital trading cards company, the latest in a series of bets aimed at turning Japanese startups into global unicorns.\n\nElection 2026\n\nOpposition to AI, data centers and Flock cameras brought people together like few other issues, and it is showing up in the midterms.\n\n| Graphics by\n\n114\n\nEarnings\n\nThe consulting company posted fourth-quarter profit of $1.99 billion, or $3.29 a share, compared with $1.41 billion, or $2.25 a share, a year earlier.\n\nPlus, FTC opens investigation of Anthropic and OpenAI\n\n1\n\nHuawei and other Chinese competitors reported higher demand for foldables after announcement of the iPhone Duo.\n\n6 [...] | Photography by Jason Henry for WSJ\n\n323\n\nCapital Account\n\nTo justify staggering investment sums, Americans will have to spen

In [27]:
from langchain_classic import hub

print("Hub imported successfully")

Hub imported successfully


In [ ]:
# prompt
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)
# print(type(prompt))


<class 'langchain_core.prompts.prompt.PromptTemplate'>
['agent_scratchpad', 'input', 'tool_names', 'tools']


In [ ]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [45]:
@tool
def get_weather_data(city:str) -> str:
    'fetch current weather data'

    return(
        print("city data printed")
    )

In [46]:
# tools
tools = [search_tool, get_weather_data]

In [39]:
# create agent

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
# agent executor
'''
1. sends input and previous messages to the agent
2. gets the next action from agent
3. Executes that tool with provided input
4. Adds thr tools observation back to the history
5. loops again with updated history until the agents says final answer
 
'''
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True   #it show all the logs if you marked it true otherwise dont show the logs just provide the output
)

In [41]:
# run

response = agent_executor.invoke({
    "input":(
        'Find the capital of India'
    )
})



> Entering new AgentExecutor chain...
I should use the tavily_search_results_json tool to find the capital of India.
Action: tavily_search_results_json
Action Input: capital of India[{'title': 'List of capitals of India - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi at the climax of the 1911 Delhi Durbar on 12 December 1911. The buildings housing the Viceroy, government, and parliament were inaugurated in early 1931.\n\n## References\n\n[edit] [...] Appearance\n\nFrom Wikipedia, the free encyclopedia\n\n|  |\n\n| This article needs more citations. Please help improve this

In [42]:
print(response)

{'input': 'Find the capital of India', 'output': 'The current capital of India is New Delhi.'}
